In [2]:
from typing import NamedTuple
from dataclasses import dataclass
from numpy.typing import NDArray
from bloodmoon.types import CoordEquatorial

class Source(NamedTuple):
    """Source info container."""
    ID: str             
    shift_x: float  # [mm]
    shift_y: float  # [mm]
    flx: float      # [Crab]

class CatalogueEntry(NamedTuple):
    """Catalogue entries data for mock sources."""
    ID: str
    name: str
    ra: float
    dec: float
    avg_flux: float
    code: int
    formula: str
    index: float
    nh: float
    energy: NDArray
    spectrum: NDArray

@dataclass(frozen=True)
class CameraPointer:
    """Instance with LEM-X coded-mask cameras pointings."""
    CAMZRA: float
    CAMZDEC: float
    CAMXRA: float
    CAMXDEC: float

    @property
    def pointings(self) -> dict[str, CoordEquatorial]:
        """
        Camera axis pointing information in equatorial frame.
        Angles are expressed in degrees.
        """
        return {
            "z": CoordEquatorial(ra=self.CAMZRA, dec=self.CAMZDEC),
            "x": CoordEquatorial(ra=self.CAMXRA, dec=self.CAMXDEC),
        }

def get_pointings(
    z_axis_RA: float = 266.4,
    z_axis_DEC: float = -28.94,
    x_axis_RA: float = 266.4,
    x_axis_DEC: float = 61.06,
) -> CameraPointer:
    """Defines LEM-X Unit pointings. Default to std directions."""
    return CameraPointer(
        z_axis_RA, z_axis_DEC, x_axis_RA, x_axis_DEC,
    )

In [3]:
import numpy as np
from bloodmoon.mask import CodedMaskCamera
from bloodmoon.coords import angle2shift

def get_intshifts(
    camera: CodedMaskCamera,
    angle_x: float,
    angle_y: float,
) -> tuple[float, float]:
    """Computes the closest integer shift coords in [mm] wrt input angural coords."""
    sx, sy = map(lambda x: angle2shift(camera, x), (angle_x, angle_y))
    int_shifts = (
        float(np.round(sx / camera.specs.mask_deltax) * camera.specs.mask_deltax),
        float(np.round(sy / camera.specs.mask_deltay) * camera.specs.mask_deltay),
    )
    return int_shifts

def get_src_phase(
    camera: CodedMaskCamera,
    centre: tuple[float, float],
    step: tuple[float, float],
    extension: int = 3,
) -> list[tuple[float, float]]:
    """
    Creates a list of coords following the given binning phase step.
    The coords (fine, coarse) are in [mm], and centered around `centre` with given `step`.
    """
    phase_x, phase_y = (
        np.arange(-extension, extension + 1) * camera.specs.mask_deltax * step[0] + centre[0],
        np.arange(-extension, extension + 1) * camera.specs.mask_deltay * step[1] + centre[1],
    )
    return [(float(sx), float(sy)) for sx, sy in zip(phase_x, phase_y)]

def get_srcs(
    camera: CodedMaskCamera,
    coords: list[tuple[float, float]],
    flxs: list[float],
    step: tuple[float, float] = (0.25, 0.0),
    extension: int = 3,
) -> list[Source]:
    """Returns list of sources with IDs and data."""
    srcs: list[Source] = []

    for idx, (tx, ty) in enumerate(coords):
        shifts_phase = get_src_phase(camera, get_intshifts(camera, tx, ty), step=step, extension=extension)
        nphase = np.arange(len(shifts_phase), dtype=int) - len(shifts_phase) // 2
        src_phase = [
            Source(f's{idx}{'+' if np.sign(phase) + 1 else '-'}{abs(phase)}', sx, sy, flxs[idx])
            for phase, (sx, sy) in zip(nphase, shifts_phase)
        ]
        srcs += src_phase
    
    return srcs

In [4]:
from pathlib import Path
from astropy.table import Table
from astropy.io import fits
from astropy.io.fits.fitsrec import FITS_rec
from bloodmoon.coords import shift2equatorial


def load_fits_data(path: Path, ext: int = 1) -> FITS_rec:
    """Loads the FITS file data from chosen extension."""
    return fits.getdata(path, ext=ext, header=False)


def gen_table(
    srcs: list[Source],
    camera: CodedMaskCamera,
    sdl: CameraPointer,
    spectr_info: FITS_rec,
) -> Table:
    """Generates the tabular data for input sources."""
    entries: list[CatalogueEntry] = []
    _flags = ('code', 'formula', 'index', 'nh', 'energy')
    flags = {key: spectr_info[key.upper()][0] for key in _flags}
    for src in srcs:
        ra, dec = shift2equatorial(sdl, camera, src.shift_x, src.shift_y)
        entry = CatalogueEntry(
            ID=src.ID,
            name=src.ID,
            ra=ra,
            dec=dec,
            avg_flux=spectr_info['AVG_FLUX'][0] * src.flx,
            spectrum=spectr_info['SPECTRUM'][0] * src.flx,
            **flags,
        )
        entries.append(entry)
    
    record = np.rec.array(
        obj=entries,
        dtype=[
            ('ID', '16U'), ('NAME', '20U'), ('RA', np.float32), ('DEC', np.float32), ('AVG_FLUX', np.float32),
            ('CODE', np.uint8), ('FORMULA', '16U'), ('INDEX', np.float32), ('NH', np.float32),
            ('ENERGY', np.float32, 513), ('SPECTRUM', np.float32, 512),
        ],
    )
    table: Table = Table(
        data=record,
        units=('', '', 'deg', 'deg', 'ph.s-1.cm-2', '', '', '', '', 'keV', 'ph.s-1.cm-2.keV-1'),
    )
    return table
        

def save_table(data: Table, save_to: str | Path) -> None:
    """Saves tabular catalogue to FITS file."""
    print("# Saving data...")
    # HDU list and Primary Header
    hdu_list = fits.HDUList([])
    primary_hdu = fits.PrimaryHDU()
    hdu_list.append(primary_hdu)
    # BinTable
    table_hdu = fits.BinTableHDU(
        data=data,
        name='SOURCES',
    )
    hdu_list.append(table_hdu)
    # save data
    hdu_list.writeto(save_to, output_verify="fix+ignore")
    hdu_list.close()
    print("# Saving completed!")
    return None

In [5]:
from bloodmoon.mask import codedmask

dirpath: str = '/mnt/dbb8f47e-da06-47bf-8ef5-038092af70f7/Edos_Magnificent_Manor/PhD_AASS/Coding/IROS_Data/Simulations'
MASK_FITS: str = f"{dirpath}/mask_NTHT_20260129_CORRECTED.fits"
wfm: CodedMaskCamera = codedmask(MASK_FITS)
sdl: CameraPointer = get_pointings()

crab_spectr: FITS_rec = load_fits_data(f"{dirpath}/crab_spectrum_2-50keV.fits")

In [6]:
srcs: list[Source] = get_srcs(
    camera=wfm,
    coords=[(0.0, 0.0), (20.0, 20.0)],
    flxs=[1.0, 1.5],
)

srcs

[Source(ID='s0-3', shift_x=-0.1875, shift_y=0.0, flx=1.0),
 Source(ID='s0-2', shift_x=-0.125, shift_y=0.0, flx=1.0),
 Source(ID='s0-1', shift_x=-0.0625, shift_y=0.0, flx=1.0),
 Source(ID='s0+0', shift_x=0.0, shift_y=0.0, flx=1.0),
 Source(ID='s0+1', shift_x=0.0625, shift_y=0.0, flx=1.0),
 Source(ID='s0+2', shift_x=0.125, shift_y=0.0, flx=1.0),
 Source(ID='s0+3', shift_x=0.1875, shift_y=0.0, flx=1.0),
 Source(ID='s1-3', shift_x=73.8125, shift_y=74.0, flx=1.5),
 Source(ID='s1-2', shift_x=73.875, shift_y=74.0, flx=1.5),
 Source(ID='s1-1', shift_x=73.9375, shift_y=74.0, flx=1.5),
 Source(ID='s1+0', shift_x=74.0, shift_y=74.0, flx=1.5),
 Source(ID='s1+1', shift_x=74.0625, shift_y=74.0, flx=1.5),
 Source(ID='s1+2', shift_x=74.125, shift_y=74.0, flx=1.5),
 Source(ID='s1+3', shift_x=74.1875, shift_y=74.0, flx=1.5)]

In [7]:
table: Table = gen_table(srcs, wfm, sdl, crab_spectr)

table

ID,NAME,RA,DEC,AVG_FLUX,CODE,FORMULA,INDEX,NH,ENERGY,SPECTRUM
,,deg,deg,ph / (s cm2),,,,,keV,ph / (keV s cm2)
str16,str20,float32,float32,float32,uint8,str16,float32,float32,float32[513],float32[512]
s0-3,s0-3,266.4,-28.992908,4.548662,1,tbabs*powerlaw,2.05,0.3,2.0 .. 50.0,2.3097186 .. 0.0034760905
s0-2,s0-2,266.4,-28.975271,4.548662,1,tbabs*powerlaw,2.05,0.3,2.0 .. 50.0,2.3097186 .. 0.0034760905
s0-1,s0-1,266.4,-28.957636,4.548662,1,tbabs*powerlaw,2.05,0.3,2.0 .. 50.0,2.3097186 .. 0.0034760905
s0+0,s0+0,266.4,-28.94,4.548662,1,tbabs*powerlaw,2.05,0.3,2.0 .. 50.0,2.3097186 .. 0.0034760905
s0+1,s0+1,266.4,-28.922363,4.548662,1,tbabs*powerlaw,2.05,0.3,2.0 .. 50.0,2.3097186 .. 0.0034760905
s0+2,s0+2,266.4,-28.904728,4.548662,1,tbabs*powerlaw,2.05,0.3,2.0 .. 50.0,2.3097186 .. 0.0034760905
s0+3,s0+3,266.4,-28.887093,4.548662,1,tbabs*powerlaw,2.05,0.3,2.0 .. 50.0,2.3097186 .. 0.0034760905
s1-3,s1-3,247.27615,-8.4755945,6.8229933,1,tbabs*powerlaw,2.05,0.3,2.0 .. 50.0,3.464578 .. 0.0052141356


In [8]:
import darksun as ds

SAVE_CATALOGUE: bool = True

if SAVE_CATALOGUE:
    save_table(table, ds.savefile_to(dirpath, 'src_phase_catalogue_2-50keV', frmt='fits'))

# Saving data...
# Saving completed!
